- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/parallel_state.py#L548

In [ ]:
"""Initialize model data parallel groups.

    Args:
        tensor_model_parallel_size (int, default = 1):
            The number of GPUs to split individual tensors across.

        pipeline_model_parallel_size (int, default = 1):
            The number of tensor parallel GPU groups to split the
            Transformer layers across. For example, if
            tensor_model_parallel_size is 4 and
            pipeline_model_parallel_size is 2, the model will be split
            into 2 groups of 4 GPUs.

        virtual_pipeline_model_parallel_size (int, optional):
            The number of stages that each pipeline group will have,
            interleaving as necessary. If None, no interleaving is
            performed. For example, if tensor_model_parallel_size is 1,
            pipeline_model_parallel_size is 4,
            virtual_pipeline_model_parallel_size is 2, and there are
            16 transformer layers in the model, the model will be
            split into 8 stages with two layers each and each GPU
            would get 2 stages as such (layer number starting with 1):

            GPU 0: [1, 2] [9, 10]
            GPU 1: [3, 4] [11, 12]
            GPU 2: [5, 6] [13, 14]
            GPU 3: [7, 8] [15, 16]

        pipeline_model_parallel_comm_backend (str, optional):
            The backend to use for pipeline parallel communication.
            If None, the default backend will be used.

        use_sharp (bool, default = False):
            Set the use of SHARP for the collective communications of
            data-parallel process groups. When `True`, run barrier
            within each data-parallel process group, which specifies
            the SHARP application target groups.

        context_parallel_size (int, default = 1):
            The number of tensor parallel GPU groups to split the
            network input sequence length across. Compute of attention
            module requires tokens of full sequence length, so GPUs
            in a context parallel group need to communicate with each
            other to exchange information of other sequence chunks.
            Each GPU and its counterparts in other tensor parallel
            groups compose a context parallel group.

            For example, assume we have 8 GPUs, if tensor model parallel
            size is 4 and context parallel size is 2, the network input
            will be split into two sequence chunks, which are processed
            by 2 different groups of 4 GPUs. One chunk is processed by
            GPU0-3, the other chunk is processed by GPU4-7. Four groups
            are build to do context parallel communications: [GPU0, GPU4],
            [GPU1, GPU5], [GPU2, GPU6], and [GPU3, GPU7].

            Context parallelism partitions sequence length, so it has no
            impact on weights, which means weights are duplicated among
            GPUs in a context parallel group. Hence, weight gradients
            all-reduce is required in backward. For simplicity, we piggyback
            GPUs of context parallelism on data parallel group for
            weight gradient all-reduce.

        expert_model_parallel_size (int, default = 1):
            The number of Mixture of Experts parallel GPUs in each expert
            parallel group.

        num_distributed_optimizer_instances (int, default = 1):
            The number of distributed optimizer replicas across the data-
            parallel domain.

        expert_tensor_parallel_size (int, default = tp_size):
            The number of GPUs to split individual tensors of expert.

        nccl_communicator_config_path (str, default = None):
            Path to the yaml file of NCCL communicator configurations.
            `min_ctas`, `max_ctas`, and `cga_cluster_size` can be set
            for each communicator.

        distributed_timeout_minutes (int, default = 30): Timeout, in
            minutes,for operations executed against distributed
            process groups. See PyTorch documentation at
            https://pytorch.org/docs/stable/distributed.html for
            caveats.

        order (str, default=tp-dp-pp):
            The rank initialization order of parallelism. Now we support
            tp-dp-pp and tp-pp-dp orders.

        get_embedding_ranks (Callable[[List[int], Optional[int]], List[int]], optional, default=None):
            A function that takes in a list of ranks for a pipeline group and returns
            those ranks that should have embeddings.

        get_position_embedding_ranks (Callable[[List[int], Optional[int]], List[int]], optional, default=None):
            A function that takes in a list of ranks for a pipeline group, and returns
            those ranks that should have position embeddings.

        create_gloo_process_groups (bool, default = True):
            Create Gloo process groups if set to True. If set to False, Gloo process groups are
            not created and calls to get Gloo process groups will result in assertion errors.

        high_priority_stream_groups (List[str], default = None):
            Specify which communicator groups should use high priority streams during creation.
            Assigning high priority to communication streams ensures that communication kernels
            are scheduled with higher priority, minimizing the exposed communication when it is
            overlapped with other computation kernels.
            Example: initialize_parallel_groups(..., high_priority_stream_groups=['dp_cp','ep_dp'])

        sharp_enabled_group (str, default = None):
            Specify which communicator group should use SHARP communication.
            This option is only valid when use_sharp is True.
            By default (None), it is enabled from dp group.
            Available options (choose one): [dp, dp_replica]

        create_all_gather_group (bool, default = False):
            Create a separate process group for all-gather operations to avoid
            head-of-line blocking with reduce-scatter operations. When enabled,
            creates an additional NCCL communicator with identical ranks as the
            dp-cp group but with independent progress engines for better communication
            overlap.

    Let's say we have a total of 16 GPUs denoted by g0 ... g15 and we
    use 2 GPUs to parallelize the model tensor, and 4 GPUs to parallelize
    the model pipeline. The present function will
    create 8 tensor model-parallel groups, 4 pipeline model-parallel groups
    and 8 data-parallel groups as:
        8 data_parallel groups:
            [g0, g2], [g1, g3], [g4, g6], [g5, g7], [g8, g10], [g9, g11], [g12, g14], [g13, g15]
        8 tensor model-parallel groups:
            [g0, g1], [g2, g3], [g4, g5], [g6, g7], [g8, g9], [g10, g11], [g12, g13], [g14, g15]
        4 pipeline model-parallel groups:
            [g0, g4, g8, g12], [g1, g5, g9, g13], [g2, g6, g10, g14], [g3, g7, g11, g15]
    Note that for efficiency, the caller should make sure adjacent ranks
    are on the same DGX box. For example if we are using 2 DGX-1 boxes
    with a total of 16 GPUs, rank 0 to 7 belong to the first box and
    ranks 8 to 15 belong to the second box.
    """

# initialize_model_parallel
- 
- 它定义了如何根据大规模模型训练的需求，<font color='red'>将多张 GPU 卡划分为不同的逻辑组（Groups）</font>，以支持张量并行、流水线并行、数据并行等多种分布式训练策略。
- 它负责在大规模分布式训练启动时，根据用户指定的并行策略（Tensor/TP, Pipeline/PP, Data/DP, Context/CP, Expert/EP），在逻辑上将物理 GPU 划分为不同的通信组（Process Groups）。

## 📚 核心参数详解
### 1. 基础并行策略
- tensor_model_parallel_size (张量并行大小)
  - 含义：<font color='red'>单个张量（如权重矩阵）被切分到多少张 GPU 上</font>。这是最细粒度的并行。
  - 示例：设为 4，意味着每 4 张卡组成一组，共同计算一个大矩阵。
- pipeline_model_parallel_size (流水线并行大小)
  - 含义：将模型的层数（Layers）切分到多少个“组”上。注意这里的“组”是指由 tensor_model_parallel_size 构成的组。
  - 逻辑：<font color='red'>总 GPU 数 = tensor_model_parallel_size × pipeline_model_parallel_size × data_parallel_size。</font>
- virtual_pipeline_model_parallel_size (虚拟流水线大小)
  - 含义：用于交错式调度。<font color='red'>如果设置此值，一个 GPU 上会存放多个不连续的层（Micro-batches 交错执行）。</font>
  - 作用：提高流水线的气泡（Bubble）效率，让 GPU 利用率更高。

### 2. 高级并行策略
- context_parallel_size (上下文/序列并行大小)
  - 含义：<font color='red'>将输入序列的长度（Sequence Length）切分到多少张 GPU 上。</font>
  - 痛点：<font color='red'>针对超长文本（如 32K tokens），显存瓶颈往往在 Attention 的 Key/Value 缓存上。切分序列长度可以大幅降低显存。</font>
  - 通信：由于 Attention 需要看到全局信息，切分后的序列块之间需要通信（通常使用 All-to-All 或 Reduce）。
  - 注意：<font color='red'>上下文并行组内的 GPU 其实共享同一份权重（类似数据并行），所以在反向传播时需要做梯度 All-Reduce。</font>
- expert_model_parallel_size (专家并行大小) & expert_tensor_parallel_size
  - 含义：专门针对 MoE (Mixture of Experts) 模型。
  - 区别：
     - expert_model_parallel_size：<font color='red'>控制有多少张卡专门用来存放专家参数。</font>
     - expert_tensor_parallel_size：<font color='red'>控制单个专家内部的张量切分度（通常等于普通的 TP Size，也可以不同）。</font>
     
### 3. 通信与后端配置
- use_sharp & sharp_enabled_group
  - 含义：SHARP 是 NVIDIA 的一种集合通信加速技术（基于集合通信的硬件卸载）。
  - 作用：<font color='red'>开启后，数据并行组内的梯度同步会利用网卡/NVSwitch 的硬件能力直接完成计算，减轻 CPU 负担。</font>
- pipeline_model_parallel_comm_backend
  - 含义：指定流水线并行使用的通信后端（通常是 NCCL 或 CUPTI）。
- nccl_communicator_config_path
  - 含义：<font color='blue'>指向一个 YAML 配置文件，用于微调 NCCL 通信参数（如 min_ctas, max_ctas），这对优化特定网络拓扑下的带宽利用率很有用。</font>
- high_priority_stream_groups
  - 含义：指定哪些通信组使用“高优先级流”。
  - 作用：在 GPU 中，通信和计算可以重叠。<font color='red'>给通信分配高优先级，可以确保数据传输尽快完成，减少计算核等待数据的时间。</font>
  
### 4. 其他配置
- distributed_timeout_minutes：分布式操作的超时时间，防止死锁。
- order：<font color='red'>指定并行维度的初始化顺序（如先 TP 再 DP 再 PP，还是先 TP 再 PP 再 DP）。这会影响 GPU Rank 的编号逻辑。</font>
- create_all_gather_group：为 All-Gather 操作创建独立的通信器，避免与 Reduce-Scatter 操作发生队头阻塞。


In [ ]:
def initialize_model_parallel(
    tensor_model_parallel_size: int = 1,
    pipeline_model_parallel_size: int = 1,
    virtual_pipeline_model_parallel_size: Optional[int] = None,
    pipeline_model_parallel_comm_backend: Optional[str] = None,
    use_sharp: bool = False,
    context_parallel_size: int = 1,
    hierarchical_context_parallel_sizes: Optional[List[int]] = None,
    hybrid_context_parallel: bool = False,
    expert_model_parallel_size: int = 1,
    num_distributed_optimizer_instances: int = 1,
    expert_tensor_parallel_size: Optional[int] = None,
    nccl_communicator_config_path: Optional[str] = None,
    distributed_timeout_minutes: int = 30,
    order: str = "tp-cp-ep-dp-pp",
    get_embedding_ranks: Optional[Callable[[List[int], Optional[int]], List[int]]] = None,
    get_position_embedding_ranks: Optional[Callable[[List[int], Optional[int]], List[int]]] = None,
    create_gloo_process_groups: bool = True,
    high_priority_stream_groups: Optional[List[str]] = None,
    sharp_enabled_group: Optional[str] = None,
    create_all_gather_group: Optional[bool] = False,
    rank_offset: int = 0,
    local_world_size: Optional[int] = None,
) -> None:
    '''
    根据用户输入的并行策略（TP/PP/DP等），
    计算各个并行维度的大小，配置高性能通信参数（如 SHARP/UCC），并生成逻辑上的“通信组”（Groups）。
    '''
    # NCCL restricts IB SHARP usage to a single communicator group—the first one created
    # with NCCL_COLLNET_ENABLE=1. After this group is created, NCCL_COLLNET_ENABLE must be
    # set to 0 for subsequent groups.
    '''
    通信优化配置：SHARP 与 NCCL
    
    这段代码处理的是 NVIDIA 针对 InfiniBand 网络的硬件加速技术——SHARP 
    (Scalable Hierarchical Aggregation and Reduction Protocol)。
    
    注释含义：NCCL（NVIDIA 的集合通信库）有一个限制：在 InfiniBand (IB) 网络上，
    SHARP 技术只能应用于第一个创建的 Communicator Group。
    创建完这个组后，必须关闭 NCCL_COLLNET_ENABLE，否则后续组无法创建。
    '''
    if "NCCL_COLLNET_ENABLE" in os.environ:
        '''
        操作：代码首先删除环境变量 NCCL_COLLNET_ENABLE，确保一个“干净”的环境
        '''
        del os.environ["NCCL_COLLNET_ENABLE"]

    '''
    如果用户启用了 use_sharp=True，代码会检查用户希望哪个组启用 SHARP（默认是 dp，即数据并行组）
    '''
    if use_sharp:
        if sharp_enabled_group is None:
            # By default, SHARP is enabled from dp group.
            sharp_enabled_group = "dp"
        else:
            # Currently, only dp and dp_replica groups are supported for SHARP.
            assert sharp_enabled_group in ["dp", "dp_replica"], "Invalid sharp_enabled_group"
            if sharp_enabled_group == "dp_replica":
                assert (
                    num_distributed_optimizer_instances > 1
                ), "dp_replica group requires num_distributed_optimizer_instances > 1"
    else:
        assert (
            sharp_enabled_group is None
        ), "sharp_enabled_group is only valid when use_sharp is True"

    if get_embedding_ranks is None:
        get_embedding_ranks = default_embedding_ranks

    if get_position_embedding_ranks is None:
        get_position_embedding_ranks = default_position_embedding_ranks

    # Get world size and rank. Ensure some consistencies.
    assert torch.distributed.is_initialized()
    '''
    world_size: 获取当前总共启动了多少个进程（即多少张 GPU
    '''
    world_size: int = (
        local_world_size if local_world_size is not None else torch.distributed.get_world_size()
    )

    '''
    model_size: 计算模型并行占用的总卡数。
    
    model_size=TP_SIZE×PP_SIZE×CP_SIZE
    
    '''
    model_size = tensor_model_parallel_size * pipeline_model_parallel_size * context_parallel_size

    if world_size % model_size != 0:
        raise RuntimeError(f"world_size ({world_size}) is not divisible by {model_size}")

    '''
    总卡数必须能被模型并行占用的卡数整除。剩下的卡数才能分配给数据并行（DP）
    data_parallel_size=world_size/model_size
    '''
    data_parallel_size: int = world_size // model_size

    '''
    **虚拟流水线检查**：
    如果使用了 `virtual_pipeline_model_parallel_size`（即 Interleaved Schedule，交错式流水线），强制要求 PP > 1。
    '''
    if virtual_pipeline_model_parallel_size is not None:
        if not pipeline_model_parallel_size > 1:
            raise RuntimeError(
                "pipeline-model-parallel size should be greater than 1 with interleaved schedule"
            )
        global _VIRTUAL_PIPELINE_MODEL_PARALLEL_RANK
        global _VIRTUAL_PIPELINE_MODEL_PARALLEL_WORLD_SIZE
        _VIRTUAL_PIPELINE_MODEL_PARALLEL_RANK = 0
        _VIRTUAL_PIPELINE_MODEL_PARALLEL_WORLD_SIZE = virtual_pipeline_model_parallel_size

    rank = torch.distributed.get_rank()

    nccl_comm_cfgs = {}
    if nccl_communicator_config_path is not None:
        try:
            import yaml
        except ImportError:
            raise RuntimeError(
                "Cannot import `yaml`. Setting custom nccl communicator configs "
                "requires the yaml package."
            )

        with open(nccl_communicator_config_path, "r") as stream:
            nccl_comm_cfgs = yaml.safe_load(stream)


            
    '''
    如果它在高优先级流组中，则将 is_high_priority_stream 标志设置为 nccl_comm_cfgs
    '''    
    # Set is_high_priority_stream flag to the nccl_comm_cfgs if it is in high_priority_stream_groups
    high_priority_stream_groups = high_priority_stream_groups or []
    for pg_name in high_priority_stream_groups:
        overwrite_nccl_comm_cfgs(nccl_comm_cfgs, pg_name, ("is_high_priority_stream", True))

    '''
    Rank 生成器
    
    这是构建并行环境的“大脑”
    
    在分布式训练中，每张卡有一个物理 ID（Rank）。
    RankGenerator 负责告诉程序：“哪些物理 ID 属于同一个 TP 组”，“哪些属于同一个 DP 组”
    
    tp: 张量并行
    pp: 流水线并行
    dp: 数据并行
    cp: 上下文并行（用于超长文本）
    ep: 专家并行（MoE 模型）

    Order 参数：order="tp-cp-ep-dp-pp" 决定了多维并行下的映射顺序。
               不同的顺序（如 PP 在前还是 DP 在前）会直接影响通信的拓扑结构和效率。
    '''
    decoder_rank_generator = RankGenerator(
        tp=tensor_model_parallel_size,
        ep=1,
        dp=data_parallel_size,
        pp=pipeline_model_parallel_size,
        cp=context_parallel_size,
        order=order,
        rank_offset=rank_offset,
    )

    # Build expert rank generator
    '''
    在 MoE 模型中，除了把不同的“专家”分到不同的卡上（专家并行 EP），单个专家内部的矩阵也可以继续切分（专家张量并行 ETP）
    
    '''
    if expert_tensor_parallel_size is None:
        '''
        如果用户没有专门指定 expert_tensor_parallel_size（专家内部的张量并行度），
        代码默认让它等于普通的 tensor_model_parallel_size（常规层的张量并行度）。
        这是一种常见的默认配置，保持并行策略的一致性。
        '''
        expert_tensor_parallel_size = tensor_model_parallel_size
    '''
    这一行计算的是“跑通一整套完整的 MoE 模型副本，最少需要多少张 GPU”
    
    expert_tensor_parallel_size：切分单个专家所需的并行度。
    expert_model_parallel_size：通常指专家并行度（EP），即把专家分散到多少张卡上。
    pipeline_model_parallel_size：流水线并行度，将模型层切分到不同阶段。
    
    '''
    expert_tensor_model_pipeline_parallel_size = (
        expert_tensor_parallel_size * expert_model_parallel_size * pipeline_model_parallel_size
    )
    '''
    计算数据并行度
    
    world_size 是你实际拥有的 GPU 总数。用总数除以“单个模型副本所需的 GPU 数”，剩下的就是数据并行度（Data Parallelism, DP）。
    
    这意味着你可以在集群中同时运行 expert_data_parallel_size 个完整的模型副本，每个副本处理不同的数据批次，以此来提高训练吞吐量。
    
    '''
    expert_data_parallel_size = world_size // expert_tensor_model_pipeline_parallel_size
    
    '''
    如果你的 GPU 总数（world_size）不能被单个模型副本所需的 GPU 数整除，说明当前的并行度配置（TP、EP、PP 的组合）是错误的，无法在当前的硬件集群上运行，因此程序会抛出错误并终止。
    
    假设你有一个 64 卡的 GPU 集群（world_size = 64），你配置了以下并行策略：
        专家并行度（EP）= 8 （把专家分到 8 张卡）
        张量并行度（TP）= 2 （每个专家内部切分到 2 张卡）
        流水线并行度（PP）= 2 （模型层切分到 2 个阶段）
        
    代码的运行过程如下：
        1.计算单副本所需卡数：2 (TP) * 8 (EP) * 2 (PP) = 32 张卡。即跑满一整套模型需要 32 张 GPU。
        2.计算数据并行度：64 (总卡数) // 32 = 2。意味着你可以在 64 张卡上同时跑 2 个完整的模型副本（数据并行度为 2）。
        3.校验：64 % 32 == 0，校验通过，配置合法
    '''
    if world_size % expert_tensor_model_pipeline_parallel_size != 0:
        raise RuntimeError(
            f"world_size ({world_size}) is not divisible by expert_tensor_model_pipeline_parallel size ({expert_tensor_model_pipeline_parallel_size})"
        )

    # TODO: support expert specific ordering 
    # 提示目前 MoE 层的并行顺序（order）暂时和非 MoE 层共用，未来可能会支持为专家层定制完全不同的并行嵌套顺序
    '''
    它的核心作用是为混合专家（MoE）层专门创建一套进程组（Process Group）的生成器，并进行严格的合法性校验。
    
    这里实例化了一个 RankGenerator 对象。你可以把它想象成一个“排兵布阵的指挥官”，它根据你传入的各个并行度（TP、EP、DP、PP等），负责计算出哪些 GPU 应该被分到一个组里进行通信。
    
    '''
    expert_decoder_rank_generator = RankGenerator(
        tp=expert_tensor_parallel_size,
        ep=expert_model_parallel_size,
        dp=expert_data_parallel_size,
        pp=pipeline_model_parallel_size,
        cp=1, # 上下文并行（Context Parallelism）设为 1，表示当前 MoE 层暂时不启用上下文并行。
        order=order, # 并行策略的嵌套顺序（例如 'tp-ep-dp-pp'），决定了 GPU 分组的层级关系。
        rank_offset=rank_offset,
    )

    '''
    这是一个非常硬核的分布式训练约束校验。它确保了当模型被切分成多个流水线阶段时，数据并行的分配不会乱套。
    
    为什么需要这个约束？ 如果 PP 不在最后，且 MoE 层和普通层的 DP 大小不一致，会导致同一个流水线阶段内的不同层（比如一层是 Attention，一层是 MoE）被分配到完全不同的 GPU 组合上，
    这会让跨层的数据传递（Residual connection 等）变得极其复杂甚至无法实现
    '''
    assert ( # 满足以下任一条件即可通过
        order.endswith("pp") # 如果流水线并行（PP）是排在最后的（即 PP-Last 策略），那么前面的 TP/EP/DP 怎么分都不会影响流水线的划分，因此是安全的。
        or pipeline_model_parallel_size == 1 # 如果没有使用流水线并行，自然不存在这个问题。
        or expert_data_parallel_size == data_parallel_size # 如果使用了 PP 且 PP 不在最后，那么要求 MoE 层的数据并行度必须等于普通层（Attention）的数据并行度。
    ), "When not using pp-last rank ordering, the data parallel size of the attention and moe layers must be the same"

    '''
    这是为了确保整个模型的流水线切分在物理 GPU 上是完全对齐的
    
    decoder_rank_generator 是普通层（非专家层）的排兵布阵指挥官，而 expert_decoder_rank_generator 是 MoE 层的。
    这段代码要求：无论当前层是普通的 Attention 还是复杂的 MoE，它们在流水线并行（PP）维度上划分出的 GPU 组必须一模一样。
    
    举例：假设 PP=2，普通层把 GPU [0,1] 分为第一阶段，[2,3] 分为第二阶段。
         那么 MoE 层也必须严格遵守 [0,1] 是第一阶段，[2,3] 是第二阶段。
         如果不对齐，流水线的前后阶段就无法传递张量，训练会直接崩溃。
    '''
    assert decoder_rank_generator.get_ranks("pp") == expert_decoder_rank_generator.get_ranks(
        "pp"
    ), f"Pipeline parallel groups are expected to be the same for Non-Expert and Expert part, \
    but got {decoder_rank_generator.get_ranks('pp')} and {expert_decoder_rank_generator.get_ranks('pp')}"

    timeout = timedelta(minutes=distributed_timeout_minutes)

    '''
    数据并行组构建:
    
    在正式把 GPU 分组去同步梯度之前，先计算好“如果我们要把优化器的状态（如动量）也切分到不同 GPU 上，每个小组应该包含多少张卡”。
    
    _GROUP_GLOO：指 GLOO 后端，通常用于 CPU 通信或特定的控制信号通信
    _WITH_CP：表示这些组包含了上下文并行的维度
    
    _INTRA_PARTIAL_...：这是关键点，指的是部分数据并行组，专门用于分布式优化器的分片。
    '''
    # Build the data-parallel groups.
    global _DATA_PARALLEL_GROUP
    global _DATA_PARALLEL_GROUP_GLOO
    global _DATA_PARALLEL_GLOBAL_RANKS
    global _DATA_PARALLEL_GROUP_WITH_CP
    global _DATA_PARALLEL_GROUP_WITH_CP_AG
    global _DATA_PARALLEL_GROUP_WITH_CP_GLOO
    global _DATA_PARALLEL_GLOBAL_RANKS_WITH_CP
    global _INTRA_PARTIAL_DATA_PARALLEL_GROUP_WITH_CP
    global _INTRA_PARTIAL_DATA_PARALLEL_GROUP_WITH_CP_GLOO
    
    '''
    确保数据并行组还没有被创建过。如果 _DATA_PARALLEL_GROUP 已经有值了，说明初始化逻辑被重复调用了，程序会报错。
    '''
    assert _DATA_PARALLEL_GROUP is None, "data parallel group is already initialized"

    '''
    背景：在分布式优化器（DistOpt）策略中，优化器的状态（如 Adam 的动量、方差）不会在每个数据并行 GPU 上都存一份，而是被切分（Shard）到不同的 GPU 上，以节省显存。
    
    data_parallel_size * context_parallel_size：这是参与梯度同步的总进程数（逻辑上的总数据并行度）。
    num_distributed_optimizer_instances：用户希望将优化器状态切分成多少份
    
    例如，如果你有 64 个数据并行 GPU，你想把优化器状态切分成 4 份，这是合法的（64 % 4 == 0）；
    如果你想切分成 3 份，这是非法的（64 % 3 != 0），因为无法均匀分配。
    
    
        '''
    assert (
        data_parallel_size * context_parallel_size
    ) % num_distributed_optimizer_instances == 0, (
        "Data parallel size should be divisible by partial DistOpt shard factor"
    )
    
    '''
    计算每个优化器分片小组内部有多少个 GPU。
    
    举例：
        - 假设 data_parallel_size = 32。
        - 假设 context_parallel_size = 1。
        - 假设我们要把优化器状态切分到 8 个实例上（num_distributed_optimizer_instances = 8）。
        - 那么 intra_partial_data_parallel_size = (32 * 1) // 8 = 4。
    结果：这意味着，每 4 个 GPU 会组成一个小组，它们共享同一份优化器分片，或者在它们内部进行特定的归约操作。


    '''
    intra_partial_data_parallel_size = (
        data_parallel_size * context_parallel_size
    ) // num_distributed_optimizer_instances

    
    
    # Set NCCL_COLLNET_ENABLE to 1 to enable SHARP for the dp group.
    '''
    核心目的是创建并初始化数据并行（DP）和上下文并行（CP）维度的通信组，
    并专门针对 NVIDIA 的 SHARP 在网计算技术进行了特殊的硬件适配
    
    它主要解决了两个关键问题：
    1. 启用 SHARP 加速：通过设置环境变量，让 NCCL 通信库能够利用 InfiniBand 交换机的硬件算力来加速集合通信。
    2. 规避硬件限制：由于 SHARP 技术的硬件限制，必须优先创建需要使用该特性的通信组
    
    启用 SHARP 加速:
    作用：如果配置指定要在数据并行（DP）组上启用 SHARP，就将环境变量 NCCL_COLLNET_ENABLE 设置为 "1"。
    '''
    if sharp_enabled_group == "dp":
        os.environ["NCCL_COLLNET_ENABLE"] = "1"

    '''
    优先创建 DP-CP 通信组（规避硬件限制）
    
    核心逻辑：这段注释揭示了非常底层的硬件约束。NCCL 的 COLLNET 特性（也就是 SHARP 在软件层面的接口）有一个硬性规定：
    只有进程启动后最初创建的那个通信组，才有资格使用 SHARP 硬件加速。
    
    代码意图：因为 dp-cp（数据并行+上下文并行）组是训练中最核心、通信量最大的部分，最需要 SHARP 的加速，
    所以代码必须把它的创建逻辑放在所有其他通信组（如普通的 DP 组、TP 组等）之前
    '''
    # In case of using SHARP, the dp-cp group requires to use NCCL COLLNET feature.
    # Due to the hardware limitation, only the initially created communication group
    # is eligible for using the NCCL COLLNET feature.
    # Therefore, dp-cp group, which potentially requires SHARP-enablement,
    # need to be created before all the other groups
    
    '''
    创建 DP-CP 核心通信组
    
    遍历所有 dp-cp 维度的进程组，并调用 create_group 创建实际的通信句柄。
    '''
    for ranks_with_cp in decoder_rank_generator.get_ranks('dp-cp'):
        
        group_with_cp = create_group(
            ranks_with_cp,
            timeout=timeout,
            pg_options=get_nccl_options("dp_cp", nccl_comm_cfgs),
            group_desc="DATA_PARALLEL_GROUP_WITH_CP",
        )
        
        '''
        AllGather 组（group_with_cp_ag）：专门用于 AllGather 操作的优化组。
        '''
        if create_all_gather_group:
            group_with_cp_ag = create_group(
                ranks_with_cp,
                timeout=timeout,
                pg_options=get_nccl_options("dp_cp", nccl_comm_cfgs),
                group_desc="DATA_PARALLEL_GROUP_WITH_CP_AG",
            )
        else:
            group_with_cp_ag = None
            
            
        if create_gloo_process_groups:
            group_with_cp_gloo = create_group(
                ranks_with_cp,
                timeout=timeout,
                backend="gloo",
                group_desc="DATA_PARALLEL_GROUP_WITH_CP_GLOO",
            )
        else:
            group_with_cp_gloo = None
            
        '''
        赋值：如果当前进程（rank）属于这个组，就将创建好的通信组句柄保存到全局变量中，供后续训练循环调用。
        '''    
        if rank in ranks_with_cp:
            _DATA_PARALLEL_GROUP_WITH_CP = group_with_cp
            _DATA_PARALLEL_GROUP_WITH_CP_AG = group_with_cp_ag
            _DATA_PARALLEL_GROUP_WITH_CP_GLOO = group_with_cp_gloo
            _DATA_PARALLEL_GLOBAL_RANKS_WITH_CP = ranks_with_cp

        '''
         创建分布式优化器的子通信组
         
         这与你之前问到的分布式优化器（DistOpt）逻辑是衔接的。
         如果启用了优化器状态分片（num_distributed_optimizer_instances > 1），
         那么在一个大的 dp-cp 组内部，还需要进一步细分出更小的“局部组”。
         
        '''
        if num_distributed_optimizer_instances > 1:
            # Create groups for intra-partial DP domain
            for i in range(num_distributed_optimizer_instances):
                
                '''
                逻辑：代码通过切片 ranks_with_cp[...] 将大的通信组切分成若干个小组，
                并分别为这些小组创建通信句柄。
                这些小组内部会进行更细粒度的梯度同步或优化器状态更新
                '''
                intra_partial_dp_ranks_with_cp = ranks_with_cp[
                    (i * intra_partial_data_parallel_size) : (
                        (i + 1) * intra_partial_data_parallel_size
                    )
                ]
                intra_partial_dp_group_with_cp = create_group(
                    intra_partial_dp_ranks_with_cp,
                    timeout=timeout,
                    pg_options=get_nccl_options("intra_dp_cp", nccl_comm_cfgs),
                    group_desc="INTRA_PARTIAL_DATA_PARALLEL_GROUP_WITH_CP",
                )
                if create_gloo_process_groups:
                    intra_partial_dp_group_with_cp_gloo = create_group(
                        intra_partial_dp_ranks_with_cp,
                        timeout=timeout,
                        backend="gloo",
                        group_desc="INTRA_PARTIAL_DATA_PARALLEL_GROUP_WITH_CP_GLOO",
                    )
                else:
                    intra_partial_dp_group_with_cp_gloo = None
                if rank in intra_partial_dp_ranks_with_cp:
                    _INTRA_PARTIAL_DATA_PARALLEL_GROUP_WITH_CP = intra_partial_dp_group_with_cp
                    _INTRA_PARTIAL_DATA_PARALLEL_GROUP_WITH_CP_GLOO = (
                        intra_partial_dp_group_with_cp_gloo
                    )
        else:
            '''
            如果没有启用优化器分片（即 num_distributed_optimizer_instances == 1），那么“局部组”就直接等同于“整体组”。
            '''
            _INTRA_PARTIAL_DATA_PARALLEL_GROUP_WITH_CP = _DATA_PARALLEL_GROUP_WITH_CP
            _INTRA_PARTIAL_DATA_PARALLEL_GROUP_WITH_CP_GLOO = _DATA_PARALLEL_GROUP_WITH_CP_GLOO

            
    '''
    核心作用是在确保 NVIDIA SHARP 技术仅作用于数据并行（DP）通信组的同时，
    完成通信组的实际创建与初始化，并处理混合上下文并行（Hybrid CP）的特殊分组逻辑。
    '''
    # Apply SHARP to the dp group.
    '''
    限制 SHARP 仅作用于 DP 组
    
    
    '''
    if sharp_enabled_group == "dp":
        if rank == 0:
            '''
            打印硬件限制警告：代码首先在主进程（rank == 0）打印了一条非常实用的日志。
            它提醒开发者，SHARP 技术的启用受限于 InfiniBand 交换机的物理型号
            （例如 NVIDIA QM1 交换机最多支持 8 个进程组，QM2 最多支持 256 个）。
            如果你的数据并行组数量超过了硬件上限，通信会自动回退到普通的非 SHARP 模式。
            '''
            logger.info(
                "The number of process groups to use SHARP with depends on the type "
                "of the network switch. Nvidia QM1 switch supports SAHRP up to 8 "
                "process groups and QM2 supports up to 256 process groups. We apply "
                "SHARP to the communications of the data-parallel domain. If the "
                "number of data-parallel process groups is larger than the max "
                "process groups that the network switch supports, the communication "
                "will fall back to non-SHARP operators. To enable SHARP, "
                "`#SBATCH_NETWORK=sharp` should be set in the sbatch script."
            )
         '''
         强制触发通信组初始化：
         
         PyTorch 的分布式通信组采用的是惰性初始化（Lazy Initialization）策略，
         即只有真正发生通信时才会去底层创建句柄。
         为了确保 SHARP 配置能准确应用到刚刚创建的 DP-CP 组上，这里主动调用了一个 barrier（屏障同步）操作，
         强制触发底层的 NCCL 初始化。
         随后的 torch.cuda.synchronize() 确保所有 GPU 都完成了这个同步操作。
         '''   
        # PyTorch is performing lazy initialization of the communicator group.
        # Therefore, we need to perform a nccl call to ensure that the communicator group is created.
        torch.distributed.barrier(
            group=get_data_parallel_group(with_context_parallel=True),
            device_ids=[torch.cuda.current_device()],
        )
        torch.cuda.synchronize()
        # Set `NCCL_COLLNET_ENABLE=0` to restrict SHARP application to the dp group.
        if "NCCL_COLLNET_ENABLE" in os.environ:
            '''
            关闭全局 SHARP 开关：
            
            在之前的代码中，为了抢占硬件资源，代码将 NCCL_COLLNET_ENABLE 设为了 "1"。
            现在 DP-CP 组已经成功创建并触发了初始化，为了避免后续创建的其他通信组（如 TP、PP 组）
            也错误地尝试使用 SHARP 硬件，
            这里必须把这个环境变量删掉（相当于设为 0），将 SHARP 严格限制在 DP 组内。
            '''
            del os.environ["NCCL_COLLNET_ENABLE"]

    '''
    处理混合上下文并行（Hybrid CP）
    
    背景：上下文并行（CP）通常用于处理超长序列。混合上下文并行（Hybrid CP）是一种更高级的并行策略。
    
    偶数限制：代码断言 len(ranks_with_cp) % 2 == 0，这是因为混合 CP 的底层通信算法（通常涉及 GPU 之间的两两配对通信）要求参与并行的 GPU 数量必须是偶数。

    创建特殊组：调用 create_hybrid_dp_cp_groups 为混合 CP 场景创建专属的通信拓扑，并更新到全局变量 _HYBRID_DP_CP_GROUPS 中。
    '''
    if hybrid_context_parallel:
        global _HYBRID_DP_CP_GROUPS
        for ranks_with_cp in decoder_rank_generator.get_ranks('dp-cp'):
            assert (
                len(ranks_with_cp) % 2 == 0
            ), "Hybrid context parallel requires an even number of ranks"
            _HYBRID_DP_CP_GROUPS.update(
                create_hybrid_dp_cp_groups(
                    rank, ranks_with_cp, get_nccl_options("dp_cp", nccl_comm_cfgs)
                )
            )
        # TODO: Are gloo groups needed for hybrid cp?

    '''
     创建普通的数据并行（DP）通信组
     
    解释：在处理好带有上下文并行（CP）的特殊情况后，最后创建最基础的、不带 CP 的纯数据并行（DP）通信组。
    逻辑：遍历所有 DP 维度的进程，调用 create_group 创建 NCCL 和 GLOO 通信句柄，
    并将当前进程所属的组和对应的全局 Rank 列表保存到全局变量中，供后续训练时的梯度同步使用。 
    
    '''
    for ranks in decoder_rank_generator.get_ranks('dp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("dp", nccl_comm_cfgs),
            group_desc="DATA_PARALLEL_GROUP",
        )
        if create_gloo_process_groups:
            group_gloo = create_group(
                ranks, timeout=timeout, backend="gloo", group_desc="DATA_PARALLEL_GROUP_GLOO"
            )
        else:
            group_gloo = None
        if rank in ranks:
            _DATA_PARALLEL_GROUP = group
            _DATA_PARALLEL_GROUP_GLOO = group_gloo
            _DATA_PARALLEL_GLOBAL_RANKS = ranks

    '''
    构建上下文并行组：
    
    它的主要任务是：根据当前的全局进程排名（rank），找到属于同一个“上下文并行组”的其他进程，
                 并建立底层的通信通道（Process Group）。此外，它还处理了更复杂的“分层上下文并行”情况。
    
    '''
    # Build the context-parallel groups.
    global _CONTEXT_PARALLEL_GROUP
    global _CONTEXT_PARALLEL_GLOBAL_RANKS
    assert _CONTEXT_PARALLEL_GROUP is None, 'context parallel group is already initialized'
    for ranks in decoder_rank_generator.get_ranks('cp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("cp", nccl_comm_cfgs),
            group_desc="CONTEXT_PARALLEL_GROUP",
        )
        if rank in ranks:
            _CONTEXT_PARALLEL_GROUP = group
            _CONTEXT_PARALLEL_GLOBAL_RANKS = ranks
        if hierarchical_context_parallel_sizes:
            assert np.prod(hierarchical_context_parallel_sizes) == context_parallel_size
            global _HIERARCHICAL_CONTEXT_PARALLEL_GROUPS
            hierarchical_groups, _ = create_hierarchical_groups(
                rank,
                ranks,
                hierarchical_context_parallel_sizes,
                create_gloo_process_groups=False,
                pg_options=get_nccl_options("hcp", nccl_comm_cfgs),
                timeout=timeout,
                group_desc="CONTEXT_PARALLEL_GROUP",
            )
            if rank in ranks:
                _HIERARCHICAL_CONTEXT_PARALLEL_GROUPS = hierarchical_groups

                
    '''
    构建模型并行组：
    
    '''
    # Build the model-parallel groups.
    global _MODEL_PARALLEL_GROUP
    global _MODEL_PARALLEL_GLOBAL_RANKS
    assert _MODEL_PARALLEL_GROUP is None, 'model parallel group is already initialized'
    for ranks in decoder_rank_generator.get_ranks('tp-pp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("mp", nccl_comm_cfgs),
            group_desc="MODEL_PARALLEL_GROUP",
        )
        if rank in ranks:
            _MODEL_PARALLEL_GROUP = group
            _MODEL_PARALLEL_GLOBAL_RANKS = ranks

    '''
    张量 模型并行组：
    
    '''
    # Build the tensor model-parallel groups.
    global _TENSOR_MODEL_PARALLEL_GROUP
    global _TENSOR_MODEL_PARALLEL_GLOBAL_RANKS
    assert (
        _TENSOR_MODEL_PARALLEL_GROUP is None
    ), 'tensor model parallel group is already initialized'
    for ranks in decoder_rank_generator.get_ranks('tp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("tp", nccl_comm_cfgs),
            group_desc="TENSOR_MODEL_PARALLEL_GROUP",
        )
        if rank in ranks:
            _TENSOR_MODEL_PARALLEL_GROUP = group
            _TENSOR_MODEL_PARALLEL_GLOBAL_RANKS = ranks

    '''
    流水线并行组：
    '''
    # Build the pipeline model-parallel groups and embedding groups
    # (first and last rank in each pipeline model-parallel group).
    global _PIPELINE_MODEL_PARALLEL_GROUP
    global _PIPELINE_GLOBAL_RANKS
    assert (
        _PIPELINE_MODEL_PARALLEL_GROUP is None
    ), "pipeline model parallel group is already initialized"
    global _EMBEDDING_GROUP
    global _EMBEDDING_GLOBAL_RANKS
    assert _EMBEDDING_GROUP is None, "embedding group is already initialized"
    global _POSITION_EMBEDDING_GROUP
    global _POSITION_EMBEDDING_GLOBAL_RANKS
    assert _POSITION_EMBEDDING_GROUP is None, "position embedding group is already initialized"
    if pipeline_model_parallel_comm_backend == "ucc":
        # The UCC backend provides two key benefits:
        # 1) Achieves better bandwidth utilization than NCCL when using InfiniBand links.
        # 2) Does not use GPU SM resources (Zero-SM), mitigating performance interference
        #    with overlapping compute kernels.

        # The UCC backend is recommended in the following cases:
        # 1) When the exposed pipeline-parallel (PP) communications are significant.
        #    - E.g., Pipeline parallelism with very less gradient accumulation steps.
        #    - It may provide better performance due to improved bandwidth utilization.
        # 2) When the critical-path pipeline stage has substantial PP-communication overlap.
        #    - E.g., Uneven pipeline parallelism.
        #    - It may provide better performance due to zero SM resource usage.
        if "CUDA_DEVICE_MAX_CONNECTIONS" in os.environ:
            # UCC backend requires CUDA_DEVICE_MAX_CONNECTIONS variable to be larger than 1,
            # to gurantee the overlapped UCC communications. If this environment variable is set to 1,
            # all the UCC communication will be serialized.
            assert (
                os.environ["CUDA_DEVICE_MAX_CONNECTIONS"] != "1"
            ), "UCC-backend requires CUDA_DEVICE_MAX_CONNECTIONS > 1"

        # Setting up required environment variables for ucc backend
        #
        # "TORCH_UCC_BLOCKING_WAIT=none" allows non-blocking waits of the communiction handle
        # "UCC_EC_CUDA_STREAM_TASK_MODE" controls how CUDA execution engines (EC)
        # schedule tasks on CUDA streams.
        # "UCX_TLS" controls transport layer selection
        # "NSYS_UCP_COMM_PARAMS=1" enables capturing ucx tracing in nsys profiling
        # "UCX_RNDV_THRESH" controls threshold threshold for switching between
        # eager and rendezvous (RNDV) communication protocols.
        # "UCX_NET_DEVICES" select which network interfaces UCX should use.
        # "UCC_CL_BASIC_TLS" controls which Transport Layers are used by
        # the Basic Collective libraray

        os.environ["TORCH_UCC_BLOCKING_WAIT"] = (
            os.environ["TORCH_UCC_BLOCKING_WAIT"]
            if "TORCH_UCC_BLOCKING_WAIT" in os.environ
            else "none"
        )
        os.environ["UCC_EC_CUDA_STREAM_TASK_MODE"] = (
            os.environ["UCC_EC_CUDA_STREAM_TASK_MODE"]
            if "UCC_EC_CUDA_STREAM_TASK_MODE" in os.environ
            else "driver"
        )
        os.environ["UCX_TLS"] = (
            os.environ["UCX_TLS"] if "UCX_TLS" in os.environ else "ib,cuda_copy"
        )  # cuda_ipc (i.e., NVLink-enablement) will be later supported
        os.environ["NSYS_UCP_COMM_PARAMS"] = "1"
        os.environ["UCX_RNDV_THRESH"] = "0"
        os.environ["UCX_NET_DEVICES"] = "all"
        os.environ["UCC_CL_BASIC_TLS"] = "^sharp,nccl"

    for ranks in decoder_rank_generator.get_ranks('pp'):
        group = create_group(
            ranks,
            timeout=timeout,
            backend=pipeline_model_parallel_comm_backend,
            pg_options=(
                None
                if pipeline_model_parallel_comm_backend == "ucc"
                else get_nccl_options("pp", nccl_comm_cfgs)
            ),
            group_desc="PIPELINE_MODEL_PARALLEL_GROUP",
        )
        assert (
            pipeline_model_parallel_comm_backend == None
            or pipeline_model_parallel_comm_backend == "nccl"
            or pipeline_model_parallel_comm_backend == "ucc"
        ), f'"{pipeline_model_parallel_comm_backend}" backend for PP communication is currently not supported'

        if rank in ranks:
            if _PIPELINE_MODEL_PARALLEL_GROUP is None:
                _PIPELINE_MODEL_PARALLEL_GROUP = group
                _PIPELINE_GLOBAL_RANKS = ranks
            elif isinstance(_PIPELINE_GLOBAL_RANKS[0], list):
                _PIPELINE_MODEL_PARALLEL_GROUP.append(group)
                _PIPELINE_GLOBAL_RANKS.append(ranks)
            else:
                _PIPELINE_MODEL_PARALLEL_GROUP = [_PIPELINE_MODEL_PARALLEL_GROUP, group]
                _PIPELINE_GLOBAL_RANKS = [_PIPELINE_GLOBAL_RANKS, ranks]

        embedding_ranks = get_embedding_ranks(ranks)
        group = create_group(
            embedding_ranks,
            timeout=timeout,
            pg_options=get_nccl_options("embd", nccl_comm_cfgs),
            group_desc="EMBEDDING_GROUP",
        )
        if rank in embedding_ranks:
            _EMBEDDING_GROUP = group
            _EMBEDDING_GLOBAL_RANKS = embedding_ranks

        position_embedding_ranks = get_position_embedding_ranks(ranks)
        group = create_group(
            position_embedding_ranks,
            timeout=timeout,
            pg_options=get_nccl_options("pos_embd", nccl_comm_cfgs),
            group_desc="POSITION_EMBEDDING_GROUP",
        )
        if rank in position_embedding_ranks:
            _POSITION_EMBEDDING_GROUP = group
            _POSITION_EMBEDDING_GLOBAL_RANKS = position_embedding_ranks

    '''
    张量 数据并行组：
    '''
    # Build the tensor + data parallel groups.
    global _TENSOR_AND_DATA_PARALLEL_GROUP
    global _TENSOR_AND_DATA_PARALLEL_GROUP_WITH_CP
    assert (
        _TENSOR_AND_DATA_PARALLEL_GROUP is None
    ), 'Tensor + data parallel group is already initialized'
    for ranks in decoder_rank_generator.get_ranks('tp-dp-cp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("tp_dp_cp", nccl_comm_cfgs),
            group_desc="TENSOR_AND_DATA_PARALLEL_GROUP_WITH_CP",
        )
        if rank in ranks:
            _TENSOR_AND_DATA_PARALLEL_GROUP_WITH_CP = group
    for ranks in decoder_rank_generator.get_ranks('tp-dp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("tp_dp", nccl_comm_cfgs),
            group_desc="TENSOR_AND_DATA_PARALLEL_GROUP",
        )
        if rank in ranks:
            _TENSOR_AND_DATA_PARALLEL_GROUP = group

    global _TENSOR_AND_CONTEXT_PARALLEL_GROUP
    assert (
        _TENSOR_AND_CONTEXT_PARALLEL_GROUP is None
    ), 'Tensor + context parallel group is already initialized'
    for ranks in decoder_rank_generator.get_ranks('tp-cp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("tp_cp", nccl_comm_cfgs),
            group_desc="TENSOR_AND_CONTEXT_PARALLEL_GROUP",
        )
        if rank in ranks:
            _TENSOR_AND_CONTEXT_PARALLEL_GROUP = group

    '''
    专家并行组：
    
    '''
    ### Expert-related parallel groups initialization
    # Build the expert model parallel group
    global _EXPERT_MODEL_PARALLEL_GROUP, _EXPERT_MODEL_PARALLEL_RANKS
    assert _EXPERT_MODEL_PARALLEL_GROUP is None, 'Expert parallel group is already initialized'
    for ranks in expert_decoder_rank_generator.get_ranks('ep'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("ep", nccl_comm_cfgs),
            group_desc="EXPERT_MODEL_PARALLEL_GROUP",
        )
        if rank in ranks:
            _EXPERT_MODEL_PARALLEL_GROUP = group
            _EXPERT_MODEL_PARALLEL_RANKS = ranks

    '''
    专家 张量并行组：
    
    '''
    # Build the expert tensor parallel group
    global _EXPERT_TENSOR_PARALLEL_GROUP
    assert (
        _EXPERT_TENSOR_PARALLEL_GROUP is None
    ), 'Expert tensor model parallel group is already initialized'
    for ranks in expert_decoder_rank_generator.get_ranks('tp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("ep_tp", nccl_comm_cfgs),
            group_desc="EXPERT_TENSOR_PARALLEL_GROUP",
        )
        if rank in ranks:
            _EXPERT_TENSOR_PARALLEL_GROUP = group

    '''
    张量 专家并行组：
    '''
    # Build the tensor + expert parallel groups
    global _EXPERT_TENSOR_AND_MODEL_PARALLEL_GROUP
    assert (
        _EXPERT_TENSOR_AND_MODEL_PARALLEL_GROUP is None
    ), 'Expert tensor + model parallel group is already initialized'
    for ranks in expert_decoder_rank_generator.get_ranks('tp-ep'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("tp_ep_mp", nccl_comm_cfgs),
            group_desc="EXPERT_TENSOR_AND_MODEL_PARALLEL_GROUP",
        )
        if rank in ranks:
            _EXPERT_TENSOR_AND_MODEL_PARALLEL_GROUP = group

    '''
    专家+张量+流水线并行组：
    '''
    # Build the expert+tensor+pipeline parallel groups
    global _EXPERT_TENSOR_MODEL_PIPELINE_PARALLEL_GROUP
    assert (
        _EXPERT_TENSOR_MODEL_PIPELINE_PARALLEL_GROUP is None
    ), 'The expert_tensor_model_pipeline parallel group is already initialized'
    for ranks in expert_decoder_rank_generator.get_ranks('tp-ep-pp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("tp_ep_pp", nccl_comm_cfgs),
            group_desc="EXPERT_TENSOR_MODEL_PIPELINE_PARALLEL_GROUP",
        )
        if rank in ranks:
            _EXPERT_TENSOR_MODEL_PIPELINE_PARALLEL_GROUP = group

    '''
    专家 数据并行组：
    '''
    # Build the expert data parallel group
    global _EXPERT_DATA_PARALLEL_GROUP
    assert _EXPERT_DATA_PARALLEL_GROUP is None, "Expert data group is already initialized"
    global _EXPERT_DATA_PARALLEL_GROUP_GLOO
    assert _EXPERT_DATA_PARALLEL_GROUP_GLOO is None, "Expert data group-gloo is already initialized"
    global _INTRA_PARTIAL_EXPERT_DATA_PARALLEL_GROUP
    assert (
        _INTRA_PARTIAL_EXPERT_DATA_PARALLEL_GROUP is None
    ), "Intra partial expert data group is already initialized"
    global _INTRA_PARTIAL_EXPERT_DATA_PARALLEL_GROUP_GLOO
    assert (
        _INTRA_PARTIAL_EXPERT_DATA_PARALLEL_GROUP_GLOO is None
    ), "Intra partial expert data group-gloo is already initialized"
    global _INTER_PARTIAL_EXPERT_DATA_PARALLEL_GROUP
    assert (
        _INTER_PARTIAL_EXPERT_DATA_PARALLEL_GROUP is None
    ), "Inter partial expert data group is already initialized"

    assert (
        expert_data_parallel_size % num_distributed_optimizer_instances == 0
    ), "Expert data parallel size should be divisible by partial DistOpt shard factor"
    intra_partial_expert_data_parallel_size = (
        expert_data_parallel_size // num_distributed_optimizer_instances
    )

    for ranks in expert_decoder_rank_generator.get_ranks('dp'):
        group = create_group(
            ranks,
            timeout=timeout,
            pg_options=get_nccl_options("ep_dp", nccl_comm_cfgs),
            group_desc="EXPERT_DATA_PARALLEL_GROUP",
        )
        if create_gloo_process_groups:
            group_gloo = create_group(
                ranks, backend="gloo", group_desc="EXPERT_DATA_PARALLEL_GROUP_GLOO"
            )
        else:
            group_gloo = None
        if rank in ranks:
            _EXPERT_DATA_PARALLEL_GROUP = group
            _EXPERT_DATA_PARALLEL_GROUP_GLOO = group_gloo

        if num_distributed_optimizer_instances > 1:
            # Create groups for Partial DistOpt, one for intra-partial DP domain
            # Another for inter-partial DP domain

            # Set NCCL_COLLNET_ENABLE to 1 to enable SHARP for the dp_replica group.
            if sharp_enabled_group == "dp_replica":
                os.environ["NCCL_COLLNET_ENABLE"] = "1"
            hierarchical_groups, hierarchical_groups_gloo = create_hierarchical_groups(
                rank,
                ranks,
                [intra_partial_expert_data_parallel_size, num_distributed_optimizer_instances],
                create_gloo_process_groups=create_gloo_process_groups,
                pg_options=[
                    get_nccl_options("intra_ep_dp", nccl_comm_cfgs),
                    get_nccl_options("inter_ep_dp", nccl_comm_cfgs),
                ],
                timeout=timeout,
                group_desc="EXPERT_DATA_PARALLEL_GROUP",
            )
            if rank in ranks:
                _INTRA_PARTIAL_EXPERT_DATA_PARALLEL_GROUP = hierarchical_groups[0]
                _INTRA_PARTIAL_EXPERT_DATA_PARALLEL_GROUP_GLOO = hierarchical_groups_gloo[0]
                _INTER_PARTIAL_EXPERT_DATA_PARALLEL_GROUP = hierarchical_groups[1]

            if sharp_enabled_group == "dp_replica":
                # PyTorch is performing lazy initialization of the communicator group.
                # Therefore, we need to perform a nccl call to ensure that the communicator group is created.
                if _INTER_PARTIAL_EXPERT_DATA_PARALLEL_GROUP is not None:
                    torch.distributed.barrier(
                        group=_INTER_PARTIAL_EXPERT_DATA_PARALLEL_GROUP,
                        device_ids=[torch.cuda.current_device()],
                    )
                    torch.cuda.synchronize()
                # Set NCCL_COLLNET_ENABLE to 0 to restrict SHARP application to the dp_replica group.
                if "NCCL_COLLNET_ENABLE" in os.environ:
                    del os.environ["NCCL_COLLNET_ENABLE"]
        else:
            _INTRA_PARTIAL_EXPERT_DATA_PARALLEL_GROUP = _EXPERT_DATA_PARALLEL_GROUP
            _INTRA_PARTIAL_EXPERT_DATA_PARALLEL_GROUP_GLOO = _EXPERT_DATA_PARALLEL_GROUP_GLOO
    ### End of expert related parallel groups initialization

    '''构建内部分布式优化器实例组'''
    # build the intra distributed optimizer instance group
    global _INTRA_DISTRIBUTED_OPTIMIZER_INSTANCE_GROUP
    assert (
        _INTRA_DISTRIBUTED_OPTIMIZER_INSTANCE_GROUP is None
    ), "Intra distributed optimizer instance group is already initialized"

    model_parallel_group_id = 0
    intra_dist_opt_ranks = []
    for ranks in expert_decoder_rank_generator.get_ranks('tp-ep-pp'):
        model_parallel_group_id += 1
        intra_dist_opt_ranks.extend(ranks)
        if model_parallel_group_id % intra_partial_expert_data_parallel_size == 0:
            intra_dist_opt_instance_group = create_group(
                intra_dist_opt_ranks,
                timeout=timeout,
                pg_options=get_nccl_options("intra_dist_opt_instance", nccl_comm_cfgs),
                group_desc="INTRA_DISTRIBUTED_OPTIMIZER_INSTANCE_GROUP",
            )
            if rank in intra_dist_opt_ranks:
                _INTRA_DISTRIBUTED_OPTIMIZER_INSTANCE_GROUP = intra_dist_opt_instance_group
            intra_dist_opt_ranks = []

    
    # Initialize global memory buffer
    # This isn't really "parallel state" but there isn't another good place to
    # put this. If we end up with a more generic initialization of megatron-core
    # we could stick it there
    _set_global_memory_buffer()
